# Choosing k for KMeans clustering

Sweeps k over a NEMI ensemble and scores each k with four criteria from
`metrics.kmeans_eval`:

- **AIC / BIC** — from a GMM fitted to the member embedding at that k. Lower is
  better; BIC penalises k harder, so it favours fewer clusters.
- **SSE** — within-cluster sum of squares from the KMeans labels. Falls
  monotonically with k, so read the elbow rather than the minimum.
- **Silhouette** — cluster separation from the KMeans labels, in [-1, 1]. Higher
  is better, and unlike the other three it is not monotone in k, so its peak is
  directly meaningful.

The ensemble is built once (cuML UMAP, `assess_overlap=False`); every k is then
clustered with cuML KMeans on those same member embeddings, so member spread
reflects UMAP stochasticity and k values are compared pairwise across members.
Each panel plots the ensemble mean with the member min–max range shaded.

In [ ]:
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import metrics.kmeans_eval as kmeans_eval
from nemi import NEMI, SingleNemi

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2",
                                          run_id="1_00",
                                          dataset_name="cutout_dataset.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io"
                                          )

source.print_available_channels()

In [ ]:
data_channels_engineered = ['gradb2','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']

dataset = cutouts_dataset.CutoutDataset.from_source(data_channels=data_channels_engineered, source=source, subset=False,
                                                    subsample_per_chunk=64, num_sample_chunks=1, n_workers=4)

In [ ]:
patch_size = 8
patches = dataset.get_patches(patch_size=patch_size)   # (N_patches, C_feat*p*p), coords excluded

print(patches.shape)

In [ ]:
EMBEDDING_DIMENSIONS = 3
ENSEMBLE_MEMBERS = 5
K_VALUES = [2, 4, 6, 8, 10, 12, 15, 20, 25, 30]

UMAP_PARAMS = {"n_components": EMBEDDING_DIMENSIONS, "n_neighbors": 57, "min_dist": 0.0}

In [ ]:
# One cuML UMAP per member; the k=K_VALUES[0] clustering here is discarded, the
# sweep re-clusters each member embedding at every k.
nemi = NEMI(params={
    "device": "gpu",
    "embedding_dict": UMAP_PARAMS,
    "clustering_dict": {"method": "kmeans", "n_clusters": K_VALUES[0]},
})
nemi.run(patches, n=ENSEMBLE_MEMBERS, assess_overlap=False)

embeddings = [member.embedding for member in nemi.nemi_pack]
print(f"{len(embeddings)} members | embedding {embeddings[0].shape}")

In [ ]:
def sweep_k(embeddings, k_values, device="gpu", seed=0):
    """Cluster every ensemble embedding at each k and score it.

    Returns a long DataFrame with one row per (member, k) and columns
    aic, bic, sse, silhouette.
    """
    rows = []
    for k in k_values:
        for member, embedding in enumerate(embeddings):
            nm = SingleNemi(params={"device": device,
                                    "clustering_dict": {"method": "kmeans", "n_clusters": k}})
            nm.embedding = embedding
            labels = nm.predict_clusters()

            bic, aic = kmeans_eval.bic_aic(embedding, k, random_state=seed)
            rows.append({"member": member, "k": k, "aic": aic, "bic": bic,
                         "sse": kmeans_eval.sse(embedding, labels),
                         "silhouette": kmeans_eval.silhouette(embedding, labels, seed=seed)})
    return pd.DataFrame(rows)

In [ ]:
scores = sweep_k(embeddings, K_VALUES)
scores.groupby("k").mean(numeric_only=True)

In [ ]:
METRICS = ("aic", "bic", "sse", "silhouette")


def plot_k_metrics(scores, metrics=METRICS, panel_size=4):
    """One panel per metric: ensemble mean against k, member min-max range shaded."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(panel_size * len(metrics), panel_size))
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, metrics):
        grouped = scores.groupby("k")[name]
        k = grouped.mean().index.to_numpy()
        ax.fill_between(k, grouped.min(), grouped.max(), alpha=0.2, label="member range")
        ax.plot(k, grouped.mean(), marker="o", label="ensemble mean")
        ax.set_xlabel("k")
        ax.set_ylabel(name)
        ax.set_title(name)
    axes[0].legend()
    fig.tight_layout()
    plt.show()

In [ ]:
plot_k_metrics(scores)